In [1]:
import pandas as pd
import numpy as np

In [2]:
employees = pd.read_csv("employees.csv")
print(employees.head())

  employee_id        name department      team  experience
0        E001  Employee_1          R  R_TEAM_1           2
1        E002  Employee_2          R  R_TEAM_1           5
2        E003  Employee_3          R  R_TEAM_1           3
3        E004  Employee_4          R  R_TEAM_1           2
4        E005  Employee_5          R  R_TEAM_1           1


In [3]:
employee_history = pd.read_csv("employee_history.csv")
print(employee_history.head())

  employee_id  name department      team  experience       task_type  \
0        E001     0          R  R_TEAM_1           2          Orders   
1        E001     0          R  R_TEAM_1           2   Notifications   
2        E001     0          R  R_TEAM_1           2    Unit Testing   
3        E001     0          R  R_TEAM_1           2    Landing Page   
4        E001     0          R  R_TEAM_1           2  Authentication   

   estimated_hours  actual_hours  accuracy  number_of_returns  
0                4           4.3    92.500                  1  
1                4           4.1    97.500                  2  
2               16          17.9    88.125                  1  
3                5           5.7    86.000                  1  
4                5           6.1    78.000                  3  


In [4]:
project_tasks = pd.read_csv("project_tasks.csv")
print(project_tasks.head())

  task_id project_id    feature  workflow_order       department  \
0   T0001       P001  Dashboard               1                R   
1   T0002       P001  Dashboard               2          Graphes   
2   T0003       P001  Dashboard               3  Quality_Control   
3   T0004       P001     Orders               1                R   
4   T0005       P001     Orders               2          Graphes   

   estimated_hours priority   status  
0              4.5     High  Pending  
1              5.3     High  Pending  
2              3.7   Medium  Pending  
3             10.8     High  Pending  
4              6.3     High  Pending  


In [5]:
projects = pd.read_csv("projects.csv")
print(projects.head())

  project_id           client project_name  start_date    deadline   status
0       P001           Orange    Project_1  2026-07-01  2026-08-13  Pending
1       P002  Tunisie Telecom    Project_2  2026-07-04  2026-08-14  Pending
2       P003          Ooredoo    Project_3  2026-07-07  2026-08-13  Pending
3       P004          BH Bank    Project_4  2026-07-10  2026-08-24  Pending
4       P005        Amen Bank    Project_5  2026-07-13  2026-08-14  Pending


In [6]:
import numpy as np

np.random.seed(42)

def generate_returns(row):

    ratio = row["actual_hours"] / row["estimated_hours"]

    if ratio <= 1:
        return np.random.randint(0, 1)

    elif ratio <= 1.15:
        return np.random.randint(1, 3)

    elif ratio <= 1.30:
        return np.random.randint(3, 5)

    else:
        return np.random.randint(5, 7)

employee_history["number_of_returns"] = employee_history.apply(
    generate_returns,
    axis=1
)

employee_history.to_csv(
    "employee_history.csv",
    index=False
)

employee_history.head()

,employee_id,name,department,team,experience,task_type,estimated_hours,actual_hours,accuracy,number_of_returns
0,E001,0,R,R_TEAM_1,2,Orders,4,4.3,92.500,1
1,E001,0,R,R_TEAM_1,2,Notifications,4,4.1,97.500,2
2,E001,0,R,R_TEAM_1,2,Unit Testing,16,17.9,88.125,1
3,E001,0,R,R_TEAM_1,2,Landing Page,5,5.7,86.000,1
4,E001,0,R,R_TEAM_1,2,Authentication,5,6.1,78.000,3


In [7]:
import pandas as pd

# ==========================================
# Charger les fichiers
# ==========================================

employees = pd.read_csv("employees.csv")
employee_history = pd.read_csv("employee_history.csv")

# ==========================================
# 1. Accuracy de chaque tâche
# ==========================================

employee_history["accuracy"] = (
    100
    - (
        abs(
            employee_history["estimated_hours"]
            - employee_history["actual_hours"]
        )
        / employee_history["estimated_hours"]
    ) * 100
)

employee_history["accuracy"] = employee_history["accuracy"].clip(lower=0)

# ==========================================
# 2. Accuracy moyenne par employé
# ==========================================

accuracy = (
    employee_history
    .groupby("employee_id")["accuracy"]
    .mean()
    .reset_index()
)

# ==========================================
# 3. Nombre moyen de retours
# ==========================================

returns = (
    employee_history
    .groupby("employee_id")["number_of_returns"]
    .mean()
    .reset_index()
)

# ==========================================
# 4. Fusion avec employees
# ==========================================

employee_score = employees.merge(
    accuracy,
    on="employee_id"
)

employee_score = employee_score.merge(
    returns,
    on="employee_id"
)

# ==========================================
# 5. Experience Score
# ==========================================

max_exp = employee_score["experience"].max()

employee_score["experience_score"] = (
    employee_score["experience"] / max_exp
) * 100

# ==========================================
# 6. Return Score
# ==========================================

max_returns = employee_score["number_of_returns"].max()

employee_score["return_score"] = (
    100
    - (
        employee_score["number_of_returns"]
        / max_returns
    ) * 100
)

employee_score["return_score"] = employee_score["return_score"].clip(lower=0)

# ==========================================
# 7. Performance Score
# ==========================================

employee_score["performance_score"] = (

      0.5 * employee_score["accuracy"]

    + 0.2 * employee_score["experience_score"]

    + 0.3 * employee_score["return_score"]

)

employee_score["performance_score"] = (
    employee_score["performance_score"].round(2)
)

# ==========================================
# 8. Trier les employés
# ==========================================

employee_score = employee_score.sort_values(
    by="performance_score",
    ascending=False
)

# ==========================================
# 9. Sauvegarder
# ==========================================

employee_score.to_csv(
    "employee_score.csv",
    index=False
)

employee_score.head(20)

,employee_id,name,department,team,experience,accuracy,number_of_returns,experience_score,return_score,performance_score
25,E026,Employee_26,Quality_Control,CQ_TEAM_2,8,89.720330,1.10,100.0,48.837209,79.51
27,E028,Employee_28,Quality_Control,CQ_TEAM_2,6,88.583106,0.95,75.0,55.813953,76.04
13,E014,Employee_14,Graphes,G_TEAM_1,6,87.840787,1.10,75.0,48.837209,73.57
14,E015,Employee_15,Graphes,G_TEAM_1,7,86.230350,1.25,87.5,41.860465,73.17
11,E012,Employee_12,Graphes,G_TEAM_1,6,88.569092,1.20,75.0,44.186047,72.54
20,E021,Employee_21,Quality_Control,CQ_TEAM_1,6,87.125660,1.20,75.0,44.186047,71.82
26,E027,Employee_27,Quality_Control,CQ_TEAM_2,6,88.870770,1.40,75.0,34.883721,69.90
23,E024,Employee_24,Quality_Control,CQ_TEAM_1,2,90.032860,0.75,25.0,65.116279,69.55
8,E009,Employee_9,R,R_TEAM_2,5,86.744177,1.20,62.5,44.186047,69.13
22,E023,Employee_23,Quality_Control,CQ_TEAM_1,4,89.553869,1.15,50.0,46.511628,68.73


In [8]:
employee_score = employee_score.sort_values(
    by=["department", "performance_score"],
    ascending=[True, False]
)

employee_score.to_csv(
    "employee_score.csv",
    index=False
)

employee_score

,employee_id,name,department,team,experience,accuracy,number_of_returns,experience_score,return_score,performance_score
13,E014,Employee_14,Graphes,G_TEAM_1,6,87.840787,1.10,75.0,48.837209,73.57
14,E015,Employee_15,Graphes,G_TEAM_1,7,86.230350,1.25,87.5,41.860465,73.17
11,E012,Employee_12,Graphes,G_TEAM_1,6,88.569092,1.20,75.0,44.186047,72.54
18,E019,Employee_19,Graphes,G_TEAM_2,7,86.443465,1.70,87.5,20.930233,67.00
15,E016,Employee_16,Graphes,G_TEAM_2,6,85.049496,1.70,75.0,20.930233,63.80
17,E018,Employee_18,Graphes,G_TEAM_2,2,85.719993,1.30,25.0,39.534884,59.72
16,E017,Employee_17,Graphes,G_TEAM_2,1,87.490783,1.35,12.5,37.209302,57.41
12,E013,Employee_13,Graphes,G_TEAM_1,4,85.026402,1.80,50.0,16.279070,57.40
19,E020,Employee_20,Graphes,G_TEAM_2,5,86.127679,2.10,62.5,2.325581,56.26
10,E011,Employee_11,Graphes,G_TEAM_1,3,85.591209,2.15,37.5,0.000000,50.30


In [9]:
#V1

In [10]:
employee_score = pd.read_csv("employee_score.csv")

projects = pd.read_csv("projects.csv")

project_tasks = pd.read_csv("project_tasks.csv")

In [11]:
employee_score["available_at"] = 0.0

In [12]:
def get_available_employee(employee_score, department):

    # employés du département
    dept = employee_score[
        employee_score["department"] == department
    ]

    # tri :
    # 1. disponible le plus tôt
    # 2. meilleur score
    dept = dept.sort_values(
        by=["available_at", "performance_score"],
        ascending=[True, False]
    )

    return dept.iloc[0]

In [13]:
planning = []

In [14]:
for _, project in projects.iterrows():

    project_id = project["project_id"]

    project_features = project_tasks[
        project_tasks["project_id"] == project_id
    ]

In [15]:
for stage in [1,2,3]:

    stage_tasks = project_features[
        project_features["workflow_order"] == stage
    ]

In [16]:
for _, task in stage_tasks.iterrows():
    employee = get_available_employee(
    employee_score,
    task["department"]
)
    start = employee["available_at"]

    end = start + task["estimated_hours"]
    planning.append({

    "project_id":project_id,

    "task_id":task["task_id"],

    "feature":task["feature"],

    "department":task["department"],

    "employee":employee["employee_id"],

    "team":employee["team"],

    "start":start,

    "end":end

})
    employee_score.loc[
    employee_score["employee_id"] == employee["employee_id"],
    "available_at"
] = end

In [17]:
planning_df = pd.DataFrame(planning)

planning_df

,project_id,task_id,feature,department,employee,team,start,end
0,P015,T0339,Login,Quality_Control,E026,CQ_TEAM_2,0.0,2.4
1,P015,T0342,Products,Quality_Control,E028,CQ_TEAM_2,0.0,3.6
2,P015,T0345,Orders,Quality_Control,E021,CQ_TEAM_1,0.0,3.0
3,P015,T0348,Cart,Quality_Control,E027,CQ_TEAM_2,0.0,3.7
4,P015,T0351,Payment,Quality_Control,E024,CQ_TEAM_1,0.0,2.8
5,P015,T0354,Notifications,Quality_Control,E023,CQ_TEAM_1,0.0,3.6
6,P015,T0357,Profile,Quality_Control,E029,CQ_TEAM_2,0.0,2.3
7,P015,T0360,Dashboard,Quality_Control,E030,CQ_TEAM_2,0.0,3.3


In [18]:
projects = projects.sort_values("start_date")

In [19]:
projects = projects.sort_values("project_id")